# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes 
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [22]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

A line chart is ideal for showing population growth trends over time and comparing changes across continents. Using a common 1970 baseline enables fair comparison, while highlighting the key trend improves visual focus.

In [23]:

# Q1 — Relative population growth by continent
growth = df_long.groupby(['Continent', 'Year'], as_index=False)['Population'].sum()
growth['Index'] = 100 * growth['Population'] / growth.groupby('Continent')['Population'].transform('first')

# Highlight the fastest-growing continent
colors = {c: '#A7A9AC' for c in growth['Continent'].unique()}
colors['Africa'] = '#0072B2'

fig = px.line(growth, x='Year', y='Index', color='Continent',
              color_discrete_map=colors,
              title='Africa grew 3.9× since 1970 — the fastest of any continent')

fig.update_traces(line_width=2)
fig.update_traces(line_width=4, selector=dict(name='Africa'))

# Direct labels at the end of each line
label_shift = {'Oceania': 20, 'South America': 0, 'Asia': -20}
for _, row in growth[growth['Year'] == 2022].iterrows():
    fig.add_annotation(x=2022, y=row['Index'], text=row['Continent'],
                       showarrow=False, xanchor='left', xshift=8,
                       yshift=label_shift.get(row['Continent'], 0),
                       font=dict(color=colors[row['Continent']]))

fig.update_layout(template='plotly_white', showlegend=False, margin=dict(r=135))
fig.update_xaxes(title=None, showgrid=False, range=[1970, 2040])
fig.update_yaxes(title='Population index (1970 = 100)', showgrid=False, zeroline=False)
fig.show()


---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

A sorted horizontal bar chart clearly ranks countries by population increase and makes differences easy to compare. Horizontal bars accommodate country names, while contrasting colors distinguish African countries from the others.

In [24]:

# Q2 — Top 15 countries by population increase
top = df.copy()
top['Increase'] = (top['2022 Population'] - top['1970 Population']) / 1e6
top = top.nlargest(15, 'Increase')
top['Group'] = np.where(top['Continent'] == 'Africa', 'Africa', 'Other')

fig = px.bar(top, x='Increase', y='Country', color='Group',
             orientation='h',
             color_discrete_map={'Africa': '#0072B2', 'Other': '#B0B0B0'},
             title='India and China lead population gains; 4 of the top 15 are African')

fig.update_traces(texttemplate='%{x:.0f}M', textposition='outside')
fig.update_layout(template='plotly_white', height=550, legend_title_text='')
fig.update_xaxes(title='Population increase (millions)', showgrid=False, range=[0, 1000])
fig.update_yaxes(title=None, categoryorder='total ascending')
fig.show()


---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

A scatter plot is suitable for examining relationships between two numerical variables and identifying outliers. Logarithmic scales handle large value differences, while continent-based colors reveal regional patterns.

In [25]:
# Q3 — Land area vs. population in 2022
labels = np.where(df['Country'].isin(['India', 'China', 'Canada', 'Australia']),
                  df['Country'], '')
fig = px.scatter(df.assign(Label=labels), x='Area', y='2022 Population',
                 color='Continent', text='Label', hover_name='Country',
                 log_x=True, log_y=True,
                 color_discrete_sequence=['#0072B2', '#E69F00', '#CC79A7',
                                          '#56B4E9', '#999999', '#8064A2'],
                 title='Land area alone does not explain population size')
fig.update_traces(marker_size=7, marker_opacity=0.7, textposition='top center')
fig.update_layout(template='plotly_white', legend_title_text='')
fig.update_xaxes(title='Area (km², log scale)', showgrid=False)
fig.update_yaxes(title='Population (log scale)', showgrid=False)
fig.show()

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

A box plot effectively compares distributions by displaying medians, quartiles, spread and outliers. It reveals variations within and between continents, while a logarithmic scale improves readability when density values differ greatly.

In [26]:
# Q4 — Population density distributions by continent
fig = px.box(df, x='Continent', y='Density', points='outliers', log_y=True,
             title='Population density varies widely within continents')
fig.update_traces(line_color='#0072B2', marker_color='#0072B2',
                  fillcolor='rgba(0,114,178,0.15)')
fig.update_layout(template='plotly_white', xaxis_title='',
                  yaxis_title='People per km² (log scale)')
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

A choropleth map is appropriate for displaying geographically distributed data and identifying regional patterns. A diverging color scale separates population growth from decline, with color intensity representing the magnitude of change.

In [27]:
# Q5 — Population change (%) from 2000 to 2022
change = df.assign(Change=100 * (df['2022 Population'] / df['2000 Population'] - 1))
fig = px.choropleth(change, locations='CCA3', color='Change', hover_name='Country',
                    color_continuous_scale=[(0, '#E69F00'), (0.111, '#F5F5F5'),
                                            (1, '#0072B2')],
                    range_color=(-40, 320),
                    labels={'Change': 'Change (%)'},
                    title='Most countries grew; declines cluster in parts of Europe')
fig.update_geos(showframe=False, showcoastlines=False, projection_type='natural earth')
fig.update_layout(template='plotly_white', margin=dict(l=0, r=0, b=0))
fig.show()

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

A treemap effectively represents hierarchical data, showing continents and their countries within one visualization. Rectangle sizes indicate relative population shares, making part-to-whole comparisons intuitive without requiring multiple charts.

In [28]:
# Q6 — 2022 population by continent and country
population = df.assign(Group=np.where(df['Continent'] == 'Asia', 'Asia', 'Other'))
fig = px.treemap(population, path=['Continent', 'Country'],
                 values='2022 Population', color='Group',
                 color_discrete_map={'Asia': '#0072B2', 'Other': '#B0B0B0'},
                 title='Asia holds about 59% of the world population in 2022')
fig.update_traces(textinfo='label+percent root')
fig.update_layout(template='plotly_white', showlegend=False,
                  margin=dict(l=5, r=5, b=5, t=55))
fig.show()

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

A 100% stacked bar chart is suitable for comparing proportional composition across different years. Normalizing each bar to 100% highlights changes in population shares, while consistent colors allow easy comparison of continents over time.

In [29]:
# Q7 — Changing continent shares of the global population
shares = df_long.assign(Group=np.where(df_long['Continent'].isin(['Africa', 'Asia', 'Europe']),
                                       df_long['Continent'], 'Rest of the world'))
shares = shares.groupby(['Year', 'Group'], as_index=False)['Population'].sum()
shares['Share'] = 100 * shares['Population'] / shares.groupby('Year')['Population'].transform('sum')
shares['Year'] = shares['Year'].astype(str)

fig = px.bar(shares, x='Year', y='Share', color='Group',
             category_orders={'Group': ['Africa', 'Asia', 'Europe', 'Rest of the world']},
             color_discrete_map={'Africa': '#E69F00', 'Asia': '#0072B2',
                                 'Europe': '#CC79A7', 'Rest of the world': '#B0B0B0'},
             title="Africa's world population share rose as Europe's fell")
fig.update_layout(template='plotly_white', barmode='stack', legend_title_text='',
                  xaxis_title='', yaxis_title='Share of world population (%)')
fig.update_yaxes(range=[0, 100], showgrid=False)
fig.update_xaxes(showgrid=False)
fig.show()